<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 19 — Precision, Recall ו־F1
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
מסנן שמסמן הרבה הודעות כזבל עשוי לתפוס את רובן, אבל גם לחסום דואר חשוב. נבדוק שתי שאלות שונות.

## מטרות
- לחשב Precision ו־Recall.
- לחשב F1.
- להסביר את המכנים.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## הנחיות למורה
זהו פתרון מתועד למעבדה. מבדילים בין תוצאה נכונה לבין נימוק נכון, ובודקים שהקוד תואם לנתוני התלמיד. המקורות הישנים נשמרו ללא שינוי.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## פתרון חלק 1
הפתרון הבא תואם לנתוני התרגול העצמאי, ולא לנתוני הדוגמה הפתורה.

</div>

In [ ]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)

In [ ]:
# דוגמה דידקטית מומצאת חדשה
actual = ["OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK",
          "Defect", "Defect", "Defect", "Defect"]
predicted = ["OK", "OK", "OK", "OK", "OK", "Defect", "Defect", "Defect",
             "OK", "Defect", "Defect", "Defect"]

<div dir="rtl" lang="he" style="text-align: right;">

## 1. מטריצה
החיובי Defect. המטריצה בסדר OK, Defect היא [[5,3], [1,3]]: TN=5, FP=3, FN=1, TP=3. בעמודת Defect יש שישה חיזויים; בשורת Defect ארבעה פגומים אמיתיים.

</div>

In [ ]:
labels_binary = ["OK", "Defect"]
cm_binary = confusion_matrix(actual, predicted, labels=labels_binary)
table_binary = pd.DataFrame(cm_binary, index=labels_binary, columns=labels_binary)
table_binary.index.name = "Actual"
table_binary.columns.name = "Predicted"
display(table_binary)

<div dir="rtl" lang="he" style="text-align: right;">

## 2. חישוב
P=3/6=0.5; R=3/4=0.75; F1=6/(6+3+1)=0.6; Accuracy=8/12≈0.6667. מחצית המסומנים אכן פגומים; שלושה רבעים מהפגומים האמיתיים זוהו. הממוצע החשבוני 0.625 אינו F1.

</div>

In [ ]:
# החיובי הוא Defect; שורות אמת, עמודות חיזוי
tn = table_binary.loc["OK", "OK"]
fp = table_binary.loc["OK", "Defect"]
fn = table_binary.loc["Defect", "OK"]
tp = table_binary.loc["Defect", "Defect"]

# בדוגמה זו המכנים חיוביים
precision_manual = tp / (tp + fp)
recall_manual = tp / (tp + fn)
f1_manual = 2 * precision_manual * recall_manual / (precision_manual + recall_manual)
f1_counts = 2 * tp / (2 * tp + fp + fn)
print("Accuracy:", (tp + tn) / cm_binary.sum())
print("Precision:", precision_manual)
print("Recall:", recall_manual)
print("F1:", f1_manual, "F1 from counts:", f1_counts)

In [ ]:
# pos_label מגדיר מה נחשב חיובי; average="binary" כי יש שתי מחלקות
p_library = precision_score(actual, predicted, pos_label="Defect", average="binary", zero_division=0)
r_library = recall_score(actual, predicted, pos_label="Defect", average="binary", zero_division=0)
f_library = f1_score(actual, predicted, pos_label="Defect", average="binary", zero_division=0)
print("Precision:", p_library, "Recall:", r_library, "F1:", f_library)

<div dir="rtl" lang="he" style="text-align: right;">

## 3. מקרה קצה

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## 4. מכנה אפס וחוסר איזון
אם אין כלל חיזויי Defect, הביטוי Precision הוא 0/0 ואינו מוגדר מתמטית. zero_division=0 היא החלטת דיווח מפורשת של הקוד: להחזיר 0 במקרה כזה. אין פירושה ש־ 0/0 שווה לאפס.
בדוגמה הבאה יש פגומים אמיתיים, לכן Recall מוגדר ושווה לאפס. F1 לפי נוסחת הספירות מוגדר ושווה לאפס, כי FN חיובי. Accuracy יכול עדיין להיות גבוה.
אם גם אין חיוביים אמיתיים, גם Recall עלול להיות לא מוגדר; קוראים תמיד את support ואת מדיניות הדיווח. כפל רשימה כאן רק מקצר בניית דוגמה מומצאת.

</div>

In [ ]:
# דוגמה מומצאת מהתרגול בשיעור 25: תמיד חוזים תקין
actual_zero = ["OK"] * 8 + ["Defect"] * 2
predicted_zero = ["OK"] * 10
print("Accuracy:", accuracy_score(actual_zero, predicted_zero))
print("Precision (chosen zero policy):", precision_score(
    actual_zero, predicted_zero, pos_label="Defect", zero_division=0
))
print("Recall:", recall_score(actual_zero, predicted_zero, pos_label="Defect", zero_division=0))
print("F1:", f1_score(actual_zero, predicted_zero, pos_label="Defect", zero_division=0))

<div dir="rtl" lang="he" style="text-align: right;">

בדוגמה זו Precision אינו מוגדר כי אין חיזויים חיוביים. מדיניות הקוד מדווחת אפס; Recall הוא 0/2 ולכן אפס מוגדר. F1 לפי ספירות הוא 0/(0+0+2)=0. Accuracy=80% אינו מעיד על הצלחה בזיהוי הפגומים.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## 5. חיבור לאותו מודל Penguins
זהו קוד הכנה נתון משיעורים 24–25; אין צורך לכתוב אותו מחדש. סנפיר ומסה, seed=42, test_size=0.25, stratify, השלמה ותקנון מאימון בלבד ו־ K=3 קבוע מראש.
היום מוסיפים מדדים לתחזיות האלה; לא משנים K או מאפיינים לאחר ראיית Test.

עותק הקורס של Penguins כולל 344 רשומות ו־ 7 עמודות; מקור Gorman ו־ Palmer Station LTER. תיעוד וייחוס: [palmerpenguins](https://github.com/allisonhorst/palmerpenguins), Horst, Hill & Gorman (2020), DOI:10.5281/zenodo.3960218. זו דוגמת הוראה, ולא אישור התאמת מאגר לפרויקט גמר.

</div>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
display(df.head())
display(df.isna().sum())

model_df = df.dropna(subset=["species"]).copy()
features = ["flipper_length_mm","body_mass_g"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Train:", len(X_train), "Test:", len(X_test))

training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)

scaler = StandardScaler().set_output(transform="pandas")
scaler.fit(X_train_ready)
X_train_scaled = scaler.transform(X_train_ready)
X_test_scaled = scaler.transform(X_test_ready)
display(X_train_scaled.head())

model = KNeighborsClassifier(n_neighbors=3, weights="uniform", metric="euclidean")
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
print(predictions[:5])

comparison = pd.DataFrame(
    {"actual": y_test, "predicted": predictions}, index=y_test.index
)
comparison["correct"] = comparison["actual"] == comparison["predicted"]
display(comparison.head(10))
correct_count = comparison["correct"].sum()
accuracy = accuracy_score(y_test, predictions)
print("Correct:", correct_count, "out of", len(y_test))
print("Accuracy:", accuracy, "Percent:", 100 * accuracy)

In [ ]:
labels_species = ["Adelie", "Chinstrap", "Gentoo"]
cm_species = confusion_matrix(y_test, predictions, labels=labels_species)
table_species = pd.DataFrame(cm_species, index=labels_species, columns=labels_species)
table_species.index.name = "Actual"
table_species.columns.name = "Predicted"
display(table_species)

<div dir="rtl" lang="he" style="text-align: right;">

## 4. Adelie מול היתר
TP הוא תא Adelie→Adelie, FP הוא יתר עמודתו ו־ FN יתר שורתו. אין לספור כל טעות במטריצה כ־ FN של Adelie.

</div>

In [ ]:
# מחלקת מטרה מול כל היתר; אין מאמנים כאן מודל בינרי חדש
target = "Adelie"
tp_target = table_species.loc[target, target]
support_target = table_species.loc[target].sum()
predicted_target = table_species[target].sum()
fn_target = support_target - tp_target
fp_target = predicted_target - tp_target
tn_target = cm_species.sum() - tp_target - fn_target - fp_target

# במחלקה ובחלוקה הנוכחית שני המכנים אינם אפס
p_target = tp_target / predicted_target
r_target = tp_target / support_target
f_target = 2 * tp_target / (2 * tp_target + fp_target + fn_target)
print("Target:", target, "TP:", tp_target, "FP:", fp_target, "FN:", fn_target, "TN:", tn_target)
print("Support:", support_target, "Predicted target:", predicted_target)
print("Precision:", p_target, "Recall:", r_target, "F1:", f_target)

<div dir="rtl" lang="he" style="text-align: right;">

## 5. דוח ופירוש

</div>

In [ ]:
# קטע נתון לקריאת דוח; output_dict מאפשר להציגו בטבלה
report = classification_report(
    y_test, predictions, labels=labels_species,
    output_dict=True, zero_division=0
)
report_table = pd.DataFrame(report).T
display(report_table.loc[labels_species, ["precision", "recall", "f1-score", "support"]])
display(report_table.loc[["macro avg", "weighted avg"], ["precision", "recall", "f1-score"]])
print("Overall Accuracy:", accuracy_score(y_test, predictions))

<div dir="rtl" lang="he" style="text-align: right;">

בתוצאת ההרצה שנבדקה, המטריצה היא [[24,13,1], [12,5,0], [0,0,31]]. עבור Adelie: TP=24, FP=12, FN=14, TN=36. P=24/36≈0.6667, R=24/38≈0.6316, F1=48/74≈0.6486; support=38. תמיד קוראים את השורה והעמודה הרלוונטיות למחלקת המטרה; TN כאן כולל כל תא שמחוץ לשתיהן.
יש לבדוק את המספרים מול פלט ההרצה אם גרסת הנתונים/הספרייה השתנתה.
macro נותן לכל מין משקל זהה; weighted משקל לפי support. ממוצע אינו מחליף בדיקת שורת Chinstrap החלשה.
בבקרת איכות שבה יקר להחמיץ פגום, נעדיף לתת חשיבות ל־ Recall של Defect, תוך דיווח Precision ומטריצה כדי להעריך עומס בדיקה. אם בדיקה נוספת יקרה במיוחד, יש לנמק גם דרישה ל־ Precision. F1 לבדו אינו משקלל עלויות כספיות ואינו מתחשב ב־ TN. אין תשובה נכונה אחת בלי הגדרת השימוש.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## תשובה לכרטיס היציאה
Precision. Recall מתחיל מכל מי שחיובי באמת.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## משוב דידקטי
בדקו בנפרד: בחירת פעולה, תקינות קוד, פירוש פלט, יחידות ומגבלה. שאלו מה ישתנה בקלט חדש. אם יש קושי, חזרו לדוגמה קטנה לפני הגדלת המאגר. זהו משוב מקומי, לא מחוון ניקוד רשמי.

</div>